# 16 — Metadata & Lineage

### Why this exists
A metric without lineage is hard to trust during an incident. This lab builds a tiny lineage graph, traces a business metric to its source, and detects a broken dependency.

### Learning objectives
represent datasets and transformations; traverse lineage; detect missing edges; distinguish metadata completeness from data correctness

### Prerequisite
Python dictionaries, sets and graph traversal

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/16-metadata-lineage.svg)

> **Question to carry through the lab:** If the revenue metric changes, can an engineer identify every upstream dataset that can affect it?

## Capability contract

**Capability:** C04 — Lineage Orchestration & Recovery  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Trace a failure through a workflow and design safe rerun or recovery behavior.


## Mental model
Lineage is a graph of dependency claims. It answers where a value came from and what may be affected by a change; it does not prove that the data itself is correct.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S05 — Metadata and lineage (DATA-1081)** → [`../work_simulations/05_acme_commerce_lineage_gap/README.md`](../work_simulations/05_acme_commerce_lineage_gap/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
nodes={
 "orders_raw":{"kind":"source"}, "payments_raw":{"kind":"source"},
 "orders_clean":{"kind":"transform"}, "payments_clean":{"kind":"transform"},
 "daily_revenue":{"kind":"metric"}
}
edges={
 "orders_clean":{"orders_raw"},
 "payments_clean":{"payments_raw"},
 "daily_revenue":{"orders_clean","payments_clean"}
}
print(nodes)

In [ ]:
def upstream(target, edges):
    seen=set(); stack=[target]
    while stack:
        x=stack.pop()
        for parent in edges.get(x,set()):
            if parent not in seen:
                seen.add(parent); stack.append(parent)
    return seen
print("revenue upstream:", sorted(upstream("daily_revenue",edges)))

In [ ]:
# A change impact query runs in the opposite direction.
def downstream(source, edges):
    out=set(); changed=True
    while changed:
        changed=False
        for child,parents in edges.items():
            if child not in out and source in parents:
                out.add(child); changed=True
            if child not in out and parents & out:
                out.add(child); changed=True
    return out
print("impact of payments_raw:", sorted(downstream("payments_raw",edges)))

In [ ]:
# Break the metadata, not the data: remove one dependency and see what becomes invisible.
broken={k:set(v) for k,v in edges.items()}
broken["daily_revenue"].remove("payments_clean")
expected={"orders_clean","payments_clean"}
visible=upstream("daily_revenue",broken)
print("missing lineage edges:", sorted(expected-visible))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## Your task

**Goal:** Detect a broken lineage graph without claiming the data itself is wrong.

**Do this:** Remove the `orders_raw → orders_clean` edge, build a completeness check, report the missing dependency, and separate lineage evidence from data-quality evidence.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## 8. Production bridge
Production systems need ownership, freshness, lineage completeness, and change impact. Treat lineage as metadata with its own quality SLO.

## 9. Independent challenge
Remove the orders_raw → orders_clean edge and create a lineage completeness check that reports the missing dependency without pretending the metric is necessarily wrong.

### Evidence to keep
lineage graph, upstream/downstream queries, missing-edge report, and a short note separating lineage evidence from data-quality evidence

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.

---

**Check your work:** [Open the reference solution](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/16.md) — only after you have attempted the exercise.